# Feature Selectivity & Single-Head Ablation — nanoTabPFN

Companion notebook to `feature_head_scoring_multi_dataset.ipynb`. Same FSS
protocol, applied to a tiny TabPFN-style model (`nanoTabPFN`) with
**3 layers × 4 heads = 12 feature heads**. Because the head pool is small,
ablation is done **one head at a time** instead of in batches.

**Pipeline**
1. Load 3 binary datasets (breast_cancer, wine[y<2], digits[n_class=2]).
2. Fit/run nanoTabPFN once per dataset; capture feature attention weights.
3. Compute **FSS** for every (layer, head, dataset) triple.
4. FSS heatmap, side-by-side per dataset.
5. Rank heads by FSS; print top-3 / bottom-3 per dataset.
6. Visualise attention maps for top-3 and bottom-3 FSS heads, per dataset.
7. Single-head ablation: zero one head at a time, record ROC-AUC drop.
8. Single-head drop heatmap per dataset.


In [ ]:
# Setup, dataset loading, model fitting, capturing feature attention ───────────

import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.stats import entropy as sp_entropy
from sklearn.datasets import load_breast_cancer, load_wine, load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from model import NanoTabPFNModel
from instrumented_model import InstrumentedNanoTabPFN

# ── Architecture / experiment constants ──────────────────────────────────────
NUM_LAYERS      = 3
NUM_HEADS       = 4
EMBEDDING_SIZE  = 96
MLP_HIDDEN_SIZE = 192
NUM_OUTPUTS     = 2
WEIGHTS_PATH    = Path('..') / 'checkpoints' / 'nanotabpfn_weights.pt'

N_TRAIN = 64; N_TEST = 32; SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)

DS_ORDER = ['breast_cancer', 'wine', 'digits']

def _auc(y_true, proba):
    return roc_auc_score(y_true, proba[:, 1])

def _load_raw():
    Xb, yb = load_breast_cancer(return_X_y=True)
    Xw, yw = load_wine(return_X_y=True)
    mask = yw < 2
    Xw, yw = Xw[mask], yw[mask]
    Xd, yd = load_digits(n_class=2, return_X_y=True)
    return {
        'breast_cancer': (Xb, yb.astype(int), load_breast_cancer().feature_names),
        'wine':          (Xw, yw.astype(int), load_wine().feature_names),
        'digits':        (Xd, yd.astype(int), np.array([f'pix_{i}' for i in range(Xd.shape[1])])),
    }

splits = {}
for name, (X, y, fnames) in _load_raw().items():
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, train_size=N_TRAIN, test_size=N_TEST,
        random_state=SEED, stratify=y,
    )
    splits[name] = {
        'X_tr': X_tr.astype(np.float32), 'X_te': X_te.astype(np.float32),
        'y_tr': y_tr, 'y_te': y_te,
        'feature_names': fnames,
    }
    print(f"  {name:<14s} n_tr={N_TRAIN} n_te={N_TEST} feat={X_tr.shape[1]} classes={len(np.unique(y))}")

base = NanoTabPFNModel(
    embedding_size=EMBEDDING_SIZE,
    num_attention_heads=NUM_HEADS,
    mlp_hidden_size=MLP_HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    num_outputs=NUM_OUTPUTS,
)
base.load_state_dict(torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True))
base.eval()
model = InstrumentedNanoTabPFN(base).eval()
print(f"\n[model] loaded {WEIGHTS_PATH.name}  layers={NUM_LAYERS}  heads={NUM_HEADS}  total feat heads={NUM_LAYERS*NUM_HEADS}")

ds_state = {}
for ds in DS_ORDER:
    s = splits[ds]
    clf = model.as_classifier(device=torch.device('cpu'))
    clf.fit(s['X_tr'], s['y_tr'])
    proba_base = clf.predict_proba(s['X_te'])
    cache = clf.last_cache
    A_mean = {li: cache['feature_attn'][li].detach().cpu().mean(dim=0).numpy()
              for li in range(NUM_LAYERS)}
    C = A_mean[0].shape[-1]
    base_auc = _auc(s['y_te'], proba_base)
    ds_state[ds] = {
        'clf': clf, 'A_mean': A_mean,
        'NL': NUM_LAYERS, 'NH': NUM_HEADS, 'C': C,
        'base_auc': base_auc,
    }
    print(f"  [{ds}]  C={C} (={C-1} feat + target)  baseline AUC={base_auc:.4f}")

In [ ]:
# Compute FSS for all 12 feature heads, all 3 datasets ───────────────
# col_profile = mean over query rows of A[h];  cp /= cp.sum()
# FSS = 1 - entropy(cp) / log(C)        →  0 = uniform, 1 = peaked
# Bookkeeping: top_col (argmax cp), target_share (cp on col C-1)

for ds in DS_ORDER:
    st = ds_state[ds]
    NL, NH, C = st['NL'], st['NH'], st['C']
    fss_matrix       = np.zeros((NL, NH))
    top_col_matrix   = np.zeros((NL, NH), dtype=int)
    tgt_share_matrix = np.zeros((NL, NH))
    max_H = np.log(C)

    for li in range(NL):
        for hi in range(NH):
            A = st['A_mean'][li][hi]
            cp = A.mean(axis=0)
            cp = cp / cp.sum()
            fss_matrix[li, hi]       = 1.0 - sp_entropy(cp) / max_H
            top_col_matrix[li, hi]   = int(np.argmax(cp))
            tgt_share_matrix[li, hi] = cp[C - 1]

    st['fss_matrix']       = fss_matrix
    st['top_col_matrix']   = top_col_matrix
    st['tgt_share_matrix'] = tgt_share_matrix

    n_target = (top_col_matrix == C - 1).sum()
    print(f"[{ds}]  FSS range=[{fss_matrix.min():.3f}, {fss_matrix.max():.3f}]  "
          f"mean={fss_matrix.mean():.3f}  target-attending heads={n_target}/{NL*NH}")

In [ ]:
# FSS heatmaps — three datasets 
fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(4.6 * len(DS_ORDER), 3.6), squeeze=False)

for idx, ds in enumerate(DS_ORDER):
    st = ds_state[ds]
    NL, NH = st['NL'], st['NH']
    M = st['fss_matrix']
    ax = axes[0, idx]
    im = ax.imshow(M, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    for li in range(NL):
        for hi in range(NH):
            v = M[li, hi]
            color = 'white' if v < 0.4 else 'black'
            ax.text(hi, li, f'{v:.2f}', ha='center', va='center',
                    fontsize=10, color=color, fontweight='bold')
    ax.set_xticks(range(NH))
    ax.set_xticklabels([f'H{h}' for h in range(NH)])
    ax.set_yticks(range(NL))
    ax.set_yticklabels([f'L{l}' for l in range(NL)])
    ax.set_xlabel('Head')
    if idx == 0:
        ax.set_ylabel('Layer')
    ax.set_title(f'FSS — {ds}', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.05, pad=0.04, label='FSS')

fig.suptitle('Feature Selectivity Score (FSS) — nanoTabPFN\n0 = uniform, 1 = peaked on one column',
             fontsize=12, y=1.04)
plt.tight_layout()
plt.savefig('../figures/nano_fss_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ranking all 12 feature heads by FSS per dataset 

SUMMARY_N = 3

for ds in DS_ORDER:
    st = ds_state[ds]
    NL, NH, C = st['NL'], st['NH'], st['C']
    fnames = splits[ds]['feature_names']

    ranked = []
    for li in range(NL):
        for hi in range(NH):
            tc = int(st['top_col_matrix'][li, hi])
            ranked.append({
                'layer': li, 'head': hi,
                'label': f'L{li}H{hi}',
                'fss': float(st['fss_matrix'][li, hi]),
                'top_col': tc,
                'top_col_name': 'TARGET' if tc == C - 1 else (
                    fnames[tc] if tc < len(fnames) else f'col_{tc}'),
                'target_share': float(st['tgt_share_matrix'][li, hi]),
            })
    ranked.sort(key=lambda h: -h['fss'])
    st['ranked_heads'] = ranked

    print(f"\n=== {ds} — top-{SUMMARY_N} and bottom-{SUMMARY_N} heads by FSS ===")
    print(f"{'Rank':>4s}  {'Head':>5s}  {'FSS':>6s}  {'TopCol':>6s}  {'TgtShare':>8s}  TopColName")
    print('-' * 64)
    for i, h in enumerate(ranked[:SUMMARY_N]):
        print(f"{i+1:4d}  {h['label']:>5s}  {h['fss']:.4f}  "
              f"{h['top_col']:6d}  {h['target_share']:.4f}    {h['top_col_name']}")
    print('  ...')
    for i, h in enumerate(ranked[-SUMMARY_N:]):
        rank = len(ranked) - SUMMARY_N + i + 1
        print(f"{rank:4d}  {h['label']:>5s}  {h['fss']:.4f}  "
              f"{h['top_col']:6d}  {h['target_share']:.4f}    {h['top_col_name']}")

In [ ]:
#Visualizing Top-3 + Bottom-3 attention maps per dataset ────────────────────────


TOP_N_VIS    = 3
BOTTOM_N_VIS = 3

for ds in DS_ORDER:
    st = ds_state[ds]
    C  = st['C']
    ranked = st['ranked_heads']
    top    = ranked[:TOP_N_VIS]
    bottom = ranked[-BOTTOM_N_VIS:][::-1]    # most uniform first

    fig, axes = plt.subplots(2, max(TOP_N_VIS, BOTTOM_N_VIS),
                             figsize=(3.4 * max(TOP_N_VIS, BOTTOM_N_VIS), 7),
                             squeeze=False)
    fig.suptitle(f'{ds} — feature attention maps (rank by FSS)\n'
                 f'rows: query column (last = TARGET) → cols: key column',
                 fontsize=11, y=1.02)

    vmax = max(
        max(st['A_mean'][h['layer']][h['head']].max() for h in top),
        max(st['A_mean'][h['layer']][h['head']].max() for h in bottom),
    )

    for col_idx, (heads, name) in enumerate([(top, 'TOP'), (bottom, 'BOTTOM')]):
        for rank, h in enumerate(heads):
            ax = axes[col_idx, rank]
            A = st['A_mean'][h['layer']][h['head']]
            im = ax.imshow(A, cmap='viridis', vmin=0, vmax=vmax, aspect='auto')
            ax.set_title(
                f"{name}#{rank+1}  {h['label']}\nFSS={h['fss']:.3f}  top_col={h['top_col_name']}",
                fontsize=9)
            ax.set_xticks([0, C - 1])
            ax.set_xticklabels(['col 0', 'TGT'], fontsize=8)
            ax.set_yticks([0, C - 1])
            ax.set_yticklabels(['col 0', 'TGT'], fontsize=8)
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(f'../figures/nano_topbot_{ds}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Single-head ablation, measuring ROC-AUC drop 

for ds in DS_ORDER:
    st  = ds_state[ds]
    s   = splits[ds]
    clf = st['clf']
    ranked = st['ranked_heads']
    base_auc = _auc(s['y_te'], clf.predict_proba(s['X_te']))

    print(f"\n[{ds}]  baseline AUC = {base_auc:.4f}")
    print(f"  {'Rank':>4s}  {'Head':>5s}  {'FSS':>6s}  {'AUC':>7s}  {'Drop':>8s}")
    print('  ' + '-' * 40)

    drops_per_head = np.zeros((NUM_LAYERS, NUM_HEADS))
    for rank, h in enumerate(ranked, start=1):
        li, hi = h['layer'], h['head']
        proba  = clf.predict_proba(s['X_te'], masked_heads={(li, hi, 'feature'): True})
        auc    = _auc(s['y_te'], proba)
        drop   = auc - base_auc
        drops_per_head[li, hi] = drop
        print(f"  {rank:>4d}  {h['label']:>5s}  {h['fss']:6.3f}  {auc:7.4f}  {drop:+8.4f}")

    st['single_head_drops'] = drops_per_head
    st['base_auc']          = base_auc


In [ ]:
# ─Single-head ablation drop heatmaps for 3 datasets

drop_matrix_all = np.stack([ds_state[ds]['single_head_drops'] for ds in DS_ORDER])
vmin = float(drop_matrix_all.min())
vmax = float(max(0.0, drop_matrix_all.max()))

fig, axes = plt.subplots(1, len(DS_ORDER), figsize=(4.6 * len(DS_ORDER), 3.6),
                         squeeze=False)
for idx, ds in enumerate(DS_ORDER):
    M  = ds_state[ds]['single_head_drops']
    ax = axes[0, idx]
    im = ax.imshow(M, cmap='RdYlGn', aspect='auto', vmin=vmin, vmax=vmax)
    for li in range(NUM_LAYERS):
        for hi in range(NUM_HEADS):
            v = M[li, hi]
            color = 'white' if v < (vmin * 0.4) else 'black'
            ax.text(hi, li, f'{v:+.3f}', ha='center', va='center',
                    fontsize=10, color=color, fontweight='bold')
    ax.set_xticks(range(NUM_HEADS)); ax.set_xticklabels([f'H{h}' for h in range(NUM_HEADS)])
    ax.set_yticks(range(NUM_LAYERS)); ax.set_yticklabels([f'L{l}' for l in range(NUM_LAYERS)])
    ax.set_xlabel('Head')
    if idx == 0:
        ax.set_ylabel('Layer')
    ax.set_title(f"{ds}\nbaseline AUC = {ds_state[ds]['base_auc']:.4f}", fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.05, pad=0.04, label='AUC drop')

fig.suptitle('Single-head ablation — ROC-AUC drop from zeroing one feature head\n'
             '(negative = degradation)',
             fontsize=12, y=1.04)
plt.tight_layout()
plt.savefig('../figures/nano_single_head_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
